# Notebook 04: Alignment Strategy Comparison

This notebook runs the full multi-strategy comparison experiment and
analyses the results across five alignment strategies:

| Strategy | Description |
|---|---|
| `baseline` | No norm constraints |
| `rule_augmented` | Norm-rule scoring over candidate pool |
| `critique_revise` | Self-critique and revision loop |
| `candidate_selector` | Best-of-N with composite reward |
| `constrained_filter` | Hard constraint filter with safe fallbacks |

> **Note:** All results are from synthetic simulations. No real data is involved.

In [ ]:
import sys
sys.path.insert(0, "../src")

from norm_dialogue_framework.config import load_config
from norm_dialogue_framework.experiments.compare_strategies import StrategyComparison
from norm_dialogue_framework.visualization.plots import (
    plot_strategy_comparison,
    plot_metric_distributions,
    plot_score_heatmap,
    plot_radar_chart,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

## 1. Run the comparison experiment

This runs 10 episodes per strategy across all 5 strategies (50 total episodes).
All outputs are deterministic given the random seed.

In [ ]:
cfg = load_config("../configs/default.yaml")

comparison = StrategyComparison(cfg=cfg)
result = comparison.run(
    n_episodes_per_strategy=10,
    output_dir="../results"
)

df = pd.DataFrame([s.model_dump() for s in result.summaries])
print(f"Total episodes: {len(df)}")
print(f"Strategies: {sorted(df['agent_strategy'].unique())}")
df.head()

## 2. Strategy leaderboard

In [ ]:
leaderboard = (
    df.groupby("agent_strategy")[
        ["composite_score", "ethical_alignment_score", "utility_score",
         "mean_coercion_risk", "mean_empathy_score", "final_trust_level"]
    ]
    .mean()
    .round(3)
    .sort_values("composite_score", ascending=False)
)
leaderboard

## 3. Strategy comparison bar chart

In [ ]:
fig = plot_strategy_comparison(
    df,
    metrics=["composite_score", "ethical_alignment_score", "utility_score"],
    use_plotly=True
)
fig.show()

## 4. Score distributions by strategy

In [ ]:
fig2 = px.violin(
    df,
    x="agent_strategy",
    y="composite_score",
    color="agent_strategy",
    box=True,
    points="all",
    title="Composite Score Distribution by Strategy"
)
fig2.update_layout(showlegend=False, yaxis_range=[0, 1])
fig2.show()

## 5. Heatmap: strategy x metric

In [ ]:
fig3 = plot_score_heatmap(df, use_plotly=True)
fig3.show()

## 6. Radar chart

In [ ]:
fig4 = plot_radar_chart(df)
fig4.show()

## 7. Performance by respondent profile

In [ ]:
profile_strategy = df.groupby(["agent_strategy", "respondent_profile"])["composite_score"].mean().reset_index()

fig5 = px.bar(
    profile_strategy,
    x="respondent_profile",
    y="composite_score",
    color="agent_strategy",
    barmode="group",
    title="Composite Score by Respondent Profile and Strategy"
)
fig5.update_layout(yaxis_range=[0, 1])
fig5.show()

## 8. Ethical vs utility trade-off

In [ ]:
fig6 = px.scatter(
    df,
    x="utility_score",
    y="ethical_alignment_score",
    color="agent_strategy",
    size="composite_score",
    hover_data=["respondent_profile", "scenario_type"],
    title="Ethical Alignment vs Utility Trade-off by Strategy"
)
fig6.update_layout(xaxis_range=[0, 1], yaxis_range=[0, 1])
fig6.show()

## 9. Statistical summary

Compute mean and standard deviation for each strategy.

In [ ]:
summary_stats = df.groupby("agent_strategy")[
    ["composite_score", "ethical_alignment_score", "utility_score",
     "mean_coercion_risk", "final_trust_level", "final_stress_level"]
].agg(["mean", "std"]).round(3)

print("Strategy-level summary statistics:")
print(summary_stats.to_string())

## 10. Key findings

Review these findings in the context of the methodology:
- Which strategy achieves the highest ethical alignment?
- Which maintains the best utility while respecting norms?
- How does the composite score trade off against each component?
- Which respondent profiles are most challenging?

> See `docs/methodology.md` for a full discussion of assumptions and limitations.